# Phase 4.2 — T-100 Carrier Investigation

This notebook investigates the structure of the T-100 Segment (All Carriers)
data, passenger-service scope, carrier identifiers, and historical carrier
coverage before selecting one airline for later forecasting work.

This notebook is exploratory. It does not construct the final route-month
modeling dataset.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

In [2]:

raw_dir = Path("../data/raw")

matching_files = list(
    raw_dir.glob("T_T100_SEGMENT_ALL_CARRIER_Jan*.zip")
)

if len(matching_files) != 1:
    raise ValueError(
        f"Expected exactly one T-100 Segment All Carriers ZIP, "
        f"found {len(matching_files)}"
    )

raw_path = matching_files[0]

In [3]:
df = pd.read_csv(raw_path)
print(df.shape)
print(df.columns.tolist())

(45204, 50)
['DEPARTURES_SCHEDULED', 'DEPARTURES_PERFORMED', 'PAYLOAD', 'SEATS', 'PASSENGERS', 'FREIGHT', 'MAIL', 'DISTANCE', 'RAMP_TO_RAMP', 'AIR_TIME', 'UNIQUE_CARRIER', 'AIRLINE_ID', 'UNIQUE_CARRIER_NAME', 'UNIQUE_CARRIER_ENTITY', 'REGION', 'CARRIER', 'CARRIER_NAME', 'CARRIER_GROUP', 'CARRIER_GROUP_NEW', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_FIPS', 'ORIGIN_STATE_NM', 'ORIGIN_COUNTRY', 'ORIGIN_COUNTRY_NAME', 'ORIGIN_WAC', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_FIPS', 'DEST_STATE_NM', 'DEST_COUNTRY', 'DEST_COUNTRY_NAME', 'DEST_WAC', 'AIRCRAFT_GROUP', 'AIRCRAFT_TYPE', 'AIRCRAFT_CONFIG', 'YEAR', 'QUARTER', 'MONTH', 'DISTANCE_GROUP', 'CLASS', 'DATA_SOURCE']


In [4]:
investigation_cols = [
    "UNIQUE_CARRIER",
    "AIRLINE_ID",
    "UNIQUE_CARRIER_ENTITY",
    "CARRIER",
    "CLASS",
    "AIRCRAFT_GROUP",
    "AIRCRAFT_TYPE",
    "AIRCRAFT_CONFIG",
    "DATA_SOURCE",
]

df[investigation_cols].nunique()

UNIQUE_CARRIER           270
AIRLINE_ID               270
UNIQUE_CARRIER_ENTITY    320
CARRIER                  270
CLASS                      4
AIRCRAFT_GROUP             9
AIRCRAFT_TYPE            143
AIRCRAFT_CONFIG            4
DATA_SOURCE                4
dtype: int64

In [5]:
df['CLASS'].value_counts(dropna=False)

CLASS
F    34526
L     5568
G     3632
P     1478
Name: count, dtype: int64

In [6]:
df.groupby('CLASS').agg(
    rows = ('CLASS', 'size'),
    total_seats = ('SEATS', 'sum'),
    total_passengers = ('PASSENGERS', 'sum'),
    total_departures = ('DEPARTURES_SCHEDULED', 'sum')
)

,rows,total_seats,total_passengers,total_departures
CLASS,,,,
F,34526,104497088.0,79882292.0,702249.0
G,3632,5438.0,532.0,37791.0
L,5568,799583.0,329657.0,0.0
P,1478,1199.0,0.0,0.0


In [7]:
df.groupby('CLASS').agg(
    rows = ('CLASS', 'size'),
    positive_seats_row = ('SEATS', lambda x: (x > 0).sum()),
    positive_passengers_row = ('PASSENGERS', lambda x: (x > 0).sum()),
    positive_departures_row = ('DEPARTURES_SCHEDULED', lambda x: (x > 0).sum())
)

,rows,positive_seats_row,positive_passengers_row,positive_departures_row
CLASS,,,,
F,34526,34322,33179,29552
G,3632,92,44,2976
L,5568,5462,4040,0
P,1478,20,0,0


In [8]:
g_passenger_rows = df.loc[
    (df['CLASS'] == 'G') & (df['PASSENGERS'] > 0),
    investigation_cols
]
print(g_passenger_rows.shape)
g_passenger_rows.head(20)

(44, 9)


,UNIQUE_CARRIER,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CLASS,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,DATA_SOURCE
13197,7S,20330,06995,7S,G,4,416,1,DU
13198,7S,20330,06995,7S,G,4,416,1,DU
13199,7S,20330,06995,7S,G,4,416,1,DU
13200,7S,20330,06995,7S,G,4,416,1,DU
13201,7S,20330,06995,7S,G,4,416,1,DU
13202,7S,20330,06995,7S,G,4,416,1,DU
13203,7S,20330,06995,7S,G,4,416,1,DU
13204,7S,20330,06995,7S,G,4,416,1,DU
13205,7S,20330,06995,7S,G,4,416,1,DU
13206,7S,20330,06995,7S,G,4,416,1,DU


In [9]:
g_passenger_rows = df.loc[
    (df['CLASS'] == 'G') & (df['PASSENGERS'] > 0),
    [
        "UNIQUE_CARRIER",
        "CARRIER_NAME",
        "ORIGIN",
        "DEST",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED",
        "AIRCRAFT_TYPE",
        "AIRCRAFT_CONFIG",
        "DATA_SOURCE",
        ]
]
print(g_passenger_rows.shape)
g_passenger_rows.head(20)

(44, 11)


,UNIQUE_CARRIER,CARRIER_NAME,ORIGIN,DEST,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,DATA_SOURCE
13197,7S,Ryan Air f/k/a Arctic Transportation,RSH,KLG,1.0,9.0,1.0,1.0,416,1,DU
13198,7S,Ryan Air f/k/a Arctic Transportation,EEK,ANI,1.0,9.0,1.0,1.0,416,1,DU
13199,7S,Ryan Air f/k/a Arctic Transportation,RDV,ANI,1.0,9.0,1.0,1.0,416,1,DU
13200,7S,Ryan Air f/k/a Arctic Transportation,CKD,RDV,1.0,9.0,1.0,1.0,416,1,DU
13201,7S,Ryan Air f/k/a Arctic Transportation,CKD,CHU,1.0,9.0,1.0,1.0,416,1,DU
13202,7S,Ryan Air f/k/a Arctic Transportation,BET,EEK,1.0,9.0,1.0,1.0,416,1,DU
13203,7S,Ryan Air f/k/a Arctic Transportation,TLT,ANI,2.0,9.0,1.0,1.0,416,1,DU
13204,7S,Ryan Air f/k/a Arctic Transportation,KKI,ANI,2.0,9.0,1.0,1.0,416,1,DU
13205,7S,Ryan Air f/k/a Arctic Transportation,OOK,TNK,2.0,9.0,1.0,1.0,416,1,DU
13206,7S,Ryan Air f/k/a Arctic Transportation,AKI,ANI,2.0,9.0,1.0,1.0,416,1,DU


In [10]:
g_passenger_rows.groupby(
    [
        "UNIQUE_CARRIER",
        "CARRIER_NAME",
        "AIRCRAFT_TYPE",
        "AIRCRAFT_CONFIG",
        "DATA_SOURCE"
    ]
    ).agg(
        rows = ('PASSENGERS', 'size'),
        total_passengers = ('PASSENGERS', 'sum'),
        min_passengers = ('PASSENGERS', 'min'),
        max_passengers = ('PASSENGERS', 'max'),
        total_seats = ('SEATS', 'sum'),
        total_departures = ('DEPARTURES_SCHEDULED', 'sum')
    )

rows  \
UNIQUE_CARRIER CARRIER_NAME                         AIRCRAFT_TYPE AIRCRAFT_CONFIG DATA_SOURCE         
7S             Ryan Air f/k/a Arctic Transportation 416           1               DU             42   
                                                    479           1               DU              2   

                                                                                               total_passengers  \
UNIQUE_CARRIER CARRIER_NAME                         AIRCRAFT_TYPE AIRCRAFT_CONFIG DATA_SOURCE                     
7S             Ryan Air f/k/a Arctic Transportation 416           1               DU                      316.0   
                                                    479           1               DU                      216.0   

                                                                                               min_passengers  \
UNIQUE_CARRIER CARRIER_NAME                         AIRCRAFT_TYPE AIRCRAFT_CONFIG DATA_SOURCE                   
7S             Ryan Air f/k/a Arctic Transportation 416           1               DU                      1.0   
                                                    479           1               DU                    104.0   

                                                                                               max_passengers  \
UNIQUE_CARRIER CARRIER_NAME                         AIRCRAFT_TYPE AIRCRAFT_CONFIG DATA_SOURCE                   
7S             Ryan Air f/k/a Arctic Transportation 416           1               DU                     51.0   
                                                    479           1               DU                    112.0   

                                                                                               total_seats  \
UNIQUE_CARRIER CARRIER_NAME                         AIRCRAFT_TYPE AIRCRAFT_CONFIG DATA_SOURCE                
7S             Ryan Air f/k/a Arctic Transportation 416           1               DU                2061.0   
                                                    479           1               DU                 342.0   

                                                                                               total_departures  
UNIQUE_CARRIER CARRIER_NAME                         AIRCRAFT_TYPE AIRCRAFT_CONFIG DATA_SOURCE                    
7S             Ryan Air f/k/a Arctic Transportation 416           1               DU                      229.0  
                                                    479           1               DU                       38.0

In [11]:
g_passenger_rows.loc[g_passenger_rows['AIRCRAFT_TYPE'] == 479]


,UNIQUE_CARRIER,CARRIER_NAME,ORIGIN,DEST,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,DATA_SOURCE
32296,7S,Ryan Air f/k/a Arctic Transportation,ANI,ANC,104.0,171.0,19.0,19.0,479,1,DU
32297,7S,Ryan Air f/k/a Arctic Transportation,ANC,ANI,112.0,171.0,19.0,19.0,479,1,DU


In [12]:
df.groupby('CLASS').agg(
    rows = ('CLASS', 'size'),
    total_dep_scheduled = ('DEPARTURES_SCHEDULED', 'sum'),
    total_dep_performed = ('DEPARTURES_PERFORMED', 'sum')
)

,rows,total_dep_scheduled,total_dep_performed
CLASS,,,
F,34526,702249.0,740838.0
G,3632,37791.0,45445.0
L,5568,0.0,14365.0
P,1478,0.0,11827.0


In [13]:
l_passenger_rows = df.loc[
    df['CLASS'] == 'L',
    [
        "UNIQUE_CARRIER",
        "CARRIER_NAME",
        "ORIGIN",
        "DEST",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED",
    ]
]
l_passenger_rows.head(20)


,UNIQUE_CARRIER,CARRIER_NAME,ORIGIN,DEST,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
3,DL,Delta Air Lines Inc.,JFK,PHL,0.0,0.0,0.0,0.0
6,DL,Delta Air Lines Inc.,YUL,HPN,0.0,0.0,0.0,0.0
7,DL,Delta Air Lines Inc.,MSP,MDW,0.0,0.0,0.0,0.0
8,DL,Delta Air Lines Inc.,IAH,ATL,0.0,0.0,0.0,0.0
9,DL,Delta Air Lines Inc.,IAH,ATL,0.0,0.0,0.0,0.0
10,DL,Delta Air Lines Inc.,GRB,ATL,0.0,0.0,0.0,0.0
11,DL,Delta Air Lines Inc.,LAX,IAH,0.0,0.0,0.0,0.0
12,DL,Delta Air Lines Inc.,LAX,IAH,0.0,0.0,0.0,0.0
13,36Q,Air Hamburg Luftverkehrsgesellschaft mbH,EWR,TEB,0.0,0.0,0.0,1.0
14,WGT,Volkswagen AirSevice GmbH dba Volkswagon,PBI,OPF,0.0,0.0,0.0,1.0


In [14]:
l_passenger_rows = df.loc[
    (df['CLASS'] == 'L') & (df['PASSENGERS'] > 0)
    ]

l_carrier_summary = l_passenger_rows.groupby(
    [
        "UNIQUE_CARRIER",
        "CARRIER_NAME"
    ]
    ).agg(
        rows = ('CLASS', 'size'),
        total_passengers = ('PASSENGERS', 'sum'),
        total_seats = ('SEATS', 'sum'),
        total_performed_departures = ('DEPARTURES_PERFORMED', 'sum')
    )
l_carrier_summary = l_carrier_summary.sort_values(
    "total_passengers",
    ascending=False
)

l_carrier_summary.head(10)

,,rows,total_passengers,total_seats,total_performed_departures
UNIQUE_CARRIER,CARRIER_NAME,,,,
XE,Delux Public Charters d/b/a JSX,87,49580.0,79355.0,2645.0
SY,Sun Country Airlines d/b/a MN Airlines,316,49046.0,80206.0,442.0
GCA,"Global Crossing Airlines, Inc.",241,35532.0,83478.0,534.0
09Q,"Swift Air, LLC d/b/a Eastern Air Lines d/b/a Eastern",305,28735.0,79000.0,763.0
KE,Korean Air Lines Co. Ltd.,6,27092.0,29598.0,126.0
DL,Delta Air Lines Inc.,332,24349.0,38552.0,451.0
WL,"Caribbean Sun Airlines, Inc. d/b/a World Atlantic Airlines",23,17024.0,73935.0,477.0
AS,Alaska Airlines Inc.,28,13511.0,19265.0,117.0
MX,Breeze Aviation Group DBA Breeze,235,11513.0,30487.0,272.0


In [15]:
fl_summary = (
    df.loc[df["CLASS"].isin(["F", "L"])]
    .groupby(
        ["UNIQUE_CARRIER", "CARRIER_NAME", "CLASS"]
    )
    .agg(
        rows=("CLASS", "size"),
        total_passengers=("PASSENGERS", "sum"),
        total_seats=("SEATS", "sum"),
        performed_departures=("DEPARTURES_PERFORMED", "sum"),
    )
)

fl_summary.loc[
    fl_summary.index.get_level_values("UNIQUE_CARRIER").isin(
        ["DL", "AS", "SY", "MX"]
    )
]

rows  \
UNIQUE_CARRIER CARRIER_NAME                           CLASS         
AS             Alaska Airlines Inc.                   F      1322   
                                                      L        43   
DL             Delta Air Lines Inc.                   F      3237   
                                                      L       540   
MX             Breeze Aviation Group DBA  Breeze      F       284   
                                                      L       235   
SY             Sun Country Airlines d/b/a MN Airlines F       124   
                                                      L       475   

                                                             total_passengers  \
UNIQUE_CARRIER CARRIER_NAME                           CLASS                     
AS             Alaska Airlines Inc.                   F             2224473.0   
                                                      L               13511.0   
DL             Delta Air Lines Inc.                   F            11990826.0   
                                                      L               24349.0   
MX             Breeze Aviation Group DBA  Breeze      F              235096.0   
                                                      L               11513.0   
SY             Sun Country Airlines d/b/a MN Airlines F              295545.0   
                                                      L               49046.0   

                                                             total_seats  \
UNIQUE_CARRIER CARRIER_NAME                           CLASS                
AS             Alaska Airlines Inc.                   F        2802420.0   
                                                      L          24783.0   
DL             Delta Air Lines Inc.                   F       14752606.0   
                                                      L          59737.0   
MX             Breeze Aviation Group DBA  Breeze      F         337661.0   
                                                      L          30487.0   
SY             Sun Country Airlines d/b/a MN Airlines F         349122.0   
                                                      L         111812.0   

                                                             performed_departures  
UNIQUE_CARRIER CARRIER_NAME                           CLASS                        
AS             Alaska Airlines Inc.                   F                   16815.0  
                                                      L                     148.0  
DL             Delta Air Lines Inc.                   F                   86241.0  
                                                      L                     677.0  
MX             Breeze Aviation Group DBA  Breeze      F                    2552.0  
                                                      L                     272.0  
SY             Sun Country Airlines d/b/a MN Airlines F                    1877.0  
                                                      L                     617.0

In [16]:
class_passengers = (
    df.loc[df["CLASS"].isin(["F", "L"]) & df["UNIQUE_CARRIER"].isin(["DL", "AS", "SY", "MX"])]
    .groupby(["CLASS", "UNIQUE_CARRIER"])
    .sum()
    .unstack()
    )
class_passengers

DEPARTURES_SCHEDULED                           \
UNIQUE_CARRIER                   AS       DL      MX      SY   
CLASS                                                          
F                           20080.0  86505.0  2548.0  1881.0   
L                               0.0      0.0     0.0     0.0   

               DEPARTURES_PERFORMED                               PAYLOAD  \
UNIQUE_CARRIER                   AS       DL      MX      SY           AS   
CLASS                                                                       
F                           16815.0  86241.0  2552.0  1877.0  726914100.0   
L                             148.0    677.0   272.0   617.0    6445700.0   

                              ... MONTH      DISTANCE_GROUP                   \
UNIQUE_CARRIER            DL  ...    MX   SY             AS     DL   MX   SY   
CLASS                         ...                                              
F               4.120823e+09  ...   284  124           4732  10052  648  400   
L               3.550101e+07  ...   235  475            104   1102  394  954   

                                                      DATA_SOURCE  \
UNIQUE_CARRIER                                                 AS   
CLASS                                                               
F               DUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDU...   
L               DUDUIUIUIUDUDUDUDUDUIUIUDUIUDUDUIUDUDUDUDUDUDU...   

                                                                   \
UNIQUE_CARRIER                                                 DL   
CLASS                                                               
F               DUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDU...   
L               DUIUDUDUDUDUDUDUDUDUDUDUDUDUIUDUDUDUDUIUDUDUDU...   

                                                                   \
UNIQUE_CARRIER                                                 MX   
CLASS                                                               
F               DUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDU...   
L               DUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDU...   

                                                                   
UNIQUE_CARRIER                                                 SY  
CLASS                                                              
F               DUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUIUIUDUDUDUDUDU...  
L               DUIUIUDUIUIUDUDUDUDUDUIUDUDUDUDUDUIUIUDUDUDUDU...  

[2 rows x 192 columns]

In [17]:
class_passengers.columns.tolist()

[('DEPARTURES_SCHEDULED', 'AS'),
 ('DEPARTURES_SCHEDULED', 'DL'),
 ('DEPARTURES_SCHEDULED', 'MX'),
 ('DEPARTURES_SCHEDULED', 'SY'),
 ('DEPARTURES_PERFORMED', 'AS'),
 ('DEPARTURES_PERFORMED', 'DL'),
 ('DEPARTURES_PERFORMED', 'MX'),
 ('DEPARTURES_PERFORMED', 'SY'),
 ('PAYLOAD', 'AS'),
 ('PAYLOAD', 'DL'),
 ('PAYLOAD', 'MX'),
 ('PAYLOAD', 'SY'),
 ('SEATS', 'AS'),
 ('SEATS', 'DL'),
 ('SEATS', 'MX'),
 ('SEATS', 'SY'),
 ('PASSENGERS', 'AS'),
 ('PASSENGERS', 'DL'),
 ('PASSENGERS', 'MX'),
 ('PASSENGERS', 'SY'),
 ('FREIGHT', 'AS'),
 ('FREIGHT', 'DL'),
 ('FREIGHT', 'MX'),
 ('FREIGHT', 'SY'),
 ('MAIL', 'AS'),
 ('MAIL', 'DL'),
 ('MAIL', 'MX'),
 ('MAIL', 'SY'),
 ('DISTANCE', 'AS'),
 ('DISTANCE', 'DL'),
 ('DISTANCE', 'MX'),
 ('DISTANCE', 'SY'),
 ('RAMP_TO_RAMP', 'AS'),
 ('RAMP_TO_RAMP', 'DL'),
 ('RAMP_TO_RAMP', 'MX'),
 ('RAMP_TO_RAMP', 'SY'),
 ('AIR_TIME', 'AS'),
 ('AIR_TIME', 'DL'),
 ('AIR_TIME', 'MX'),
 ('AIR_TIME', 'SY'),
 ('AIRLINE_ID', 'AS'),
 ('AIRLINE_ID', 'DL'),
 ('AIRLINE_ID', 'MX'),
 ('AIRL

In [18]:
class_passenger_summary = (
    df.loc[
        df["UNIQUE_CARRIER"].isin(["DL", "AS", "SY", "MX"])
        & df["CLASS"].isin(["F", "L"]),
        ["UNIQUE_CARRIER", "CLASS", "PASSENGERS"]
    ]
    .groupby(["UNIQUE_CARRIER", "CLASS"])["PASSENGERS"]
    .sum()
    .unstack("CLASS")
)

class_passenger_summary

CLASS,F,L
UNIQUE_CARRIER,,
AS,2224473.0,13511.0
DL,11990826.0,24349.0
MX,235096.0,11513.0
SY,295545.0,49046.0


In [19]:
class_passenger_summary.columns.tolist()

['F', 'L']

In [20]:
class_passenger_summary["L_share_percent"] = (
    class_passenger_summary["L"]
    / (class_passenger_summary["F"] + class_passenger_summary["L"])
    * 100
)

class_passenger_summary

CLASS,F,L,L_share_percent
UNIQUE_CARRIER,,,
AS,2224473.0,13511.0,0.603713
DL,11990826.0,24349.0,0.202652
MX,235096.0,11513.0,4.668524
SY,295545.0,49046.0,14.233105


In [21]:
f_df = df.loc[
    df["CLASS"] == "F",
    
    ]

route_month_row_counts = f_df.groupby(
    ["UNIQUE_CARRIER", "YEAR", "MONTH", "ORIGIN", "DEST"]
    ).size()

route_month_row_counts.value_counts().sort_index()

1     9641
2     3993
3     2845
4      951
5      379
6      193
7      100
8       40
9       15
10      12
11      12
12       5
13       2
14       1
Name: count, dtype: int64

In [22]:
index = route_month_row_counts.idxmax()
carrier, year, month, origin, dest = index
print(index, ":", route_month_row_counts.max())

('UA', np.int64(2024), np.int64(1), 'SAT', 'IAH') : 14


In [23]:
max_route_month_rows = f_df.loc[
    (f_df["UNIQUE_CARRIER"] == carrier) 
    & (f_df["YEAR"] == year) 
    & (f_df["MONTH"] == month) 
    & (f_df["ORIGIN"] == origin) 
    & (f_df["DEST"] == dest)
    ]

max_route_month_rows

,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,PAYLOAD,SEATS,PASSENGERS,FREIGHT,MAIL,DISTANCE,RAMP_TO_RAMP,AIR_TIME,...,DEST_WAC,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,YEAR,QUARTER,MONTH,DISTANCE_GROUP,CLASS,DATA_SOURCE
2956,0.0,1.0,30473.0,126.0,3.0,0.0,0.0,191.0,77.0,38.0,...,74,6,698,1,2024,1,1,1,F,DU
3440,0.0,1.0,34709.0,150.0,146.0,0.0,0.0,191.0,52.0,36.0,...,74,6,694,1,2024,1,1,1,F,DU
3679,0.0,1.0,36551.0,166.0,113.0,0.0,0.0,191.0,54.0,34.0,...,74,6,614,1,2024,1,1,1,F,DU
3886,0.0,1.0,39820.0,166.0,164.0,0.0,0.0,191.0,63.0,44.0,...,74,6,838,1,2024,1,1,1,F,DU
6109,0.0,1.0,100907.0,276.0,272.0,26122.0,1.0,191.0,79.0,43.0,...,74,6,627,1,2024,1,1,1,F,DU
7294,0.0,2.0,88439.0,358.0,350.0,0.0,0.0,191.0,233.0,128.0,...,74,6,888,1,2024,1,1,1,F,DU
22824,5.0,4.0,121824.0,504.0,260.0,103.0,0.0,191.0,253.0,147.0,...,74,6,612,1,2024,1,1,1,F,DU
22832,5.0,4.0,139373.0,504.0,172.0,0.0,0.0,191.0,305.0,165.0,...,74,6,698,1,2024,1,1,1,F,DU
23703,6.0,4.0,196758.0,716.0,679.0,0.0,0.0,191.0,216.0,151.0,...,74,6,839,1,2024,1,1,1,F,DU
24725,7.0,6.0,256584.0,996.0,400.0,781.0,1318.0,191.0,378.0,232.0,...,74,6,838,1,2024,1,1,1,F,DU


In [24]:
columns_to_check = [
    "AIRCRAFT_GROUP",
    "AIRCRAFT_TYPE",
    "AIRCRAFT_CONFIG",
    "DATA_SOURCE",
    "AIRLINE_ID",
    "UNIQUE_CARRIER_ENTITY",
    "CARRIER",
    "DISTANCE",
]

max_route_month_rows[columns_to_check].nunique()

AIRCRAFT_GROUP           1
AIRCRAFT_TYPE            9
AIRCRAFT_CONFIG          1
DATA_SOURCE              1
AIRLINE_ID               1
UNIQUE_CARRIER_ENTITY    3
CARRIER                  1
DISTANCE                 1
dtype: int64

In [25]:
max_route_month_rows.groupby(
    [
        "UNIQUE_CARRIER_ENTITY",
        "AIRCRAFT_TYPE"
    ]
    ).agg(
        rows = ('CLASS', 'size'),
        total_passengers = ('PASSENGERS', 'sum'),
        total_seats = ('SEATS', 'sum'),
        total_performed_departures = ('DEPARTURES_PERFORMED', 'sum')
    )

rows  total_passengers  total_seats  \
UNIQUE_CARRIER_ENTITY AIRCRAFT_TYPE                                        
0A875                 612               1             260.0        504.0   
                      614               1            5650.0       8632.0   
                      634               1            1811.0       3043.0   
                      694               1            1174.0       1500.0   
                      698               1             172.0        504.0   
                      838               1             400.0        996.0   
                      839               1             679.0        716.0   
                      888               1            3117.0       3938.0   
10876                 614               1             113.0        166.0   
                      694               1             146.0        150.0   
                      698               1               3.0        126.0   
                      838               1             164.0        166.0   
                      888               1             350.0        358.0   
10877                 627               1             272.0        276.0   

                                     total_performed_departures  
UNIQUE_CARRIER_ENTITY AIRCRAFT_TYPE                              
0A875                 612                                   4.0  
                      614                                  52.0  
                      634                                  17.0  
                      694                                  10.0  
                      698                                   4.0  
                      838                                   6.0  
                      839                                   4.0  
                      888                                  22.0  
10876                 614                                   1.0  
                      694                                   1.0  
                      698                                   1.0  
                      838                                   1.0  
                      888                                   2.0  
10877                 627                                   1.0

In [26]:
max_route_month_rows[
    [
        "UNIQUE_CARRIER",
        "AIRLINE_ID",
        "UNIQUE_CARRIER_NAME",
        "UNIQUE_CARRIER_ENTITY",
        "CARRIER",
        "CARRIER_NAME",
    ]
].drop_duplicates()

,UNIQUE_CARRIER,AIRLINE_ID,UNIQUE_CARRIER_NAME,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
2956,UA,19977,United Air Lines Inc.,10876,UA,United Air Lines Inc.
6109,UA,19977,United Air Lines Inc.,10877,UA,United Air Lines Inc.
22824,UA,19977,United Air Lines Inc.,0A875,UA,United Air Lines Inc.


In [27]:
carrier_identifier_investigation = (
    f_df[
        [
        'UNIQUE_CARRIER',
        'AIRLINE_ID',
        'UNIQUE_CARRIER_ENTITY',
        'CARRIER',
        'CARRIER_NAME'
        ]
    ]
    .groupby('UNIQUE_CARRIER')
    .nunique()
)
carrier_identifier_investigation.head()

,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
UNIQUE_CARRIER,,,,
07Q,1,1,1,1
1QQ,1,1,1,1
2E,1,1,1,1
2K,1,1,1,1
2NQ,1,1,1,1


In [28]:
carrier_identifier_investigation[
    carrier_identifier_investigation['UNIQUE_CARRIER_ENTITY'] > 1
]

,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
UNIQUE_CARRIER,,,,
3M,1,2,1,1
9E,1,2,1,1
AA,1,4,1,1
AS,1,2,1,1
B6,1,3,1,1
DL,1,4,1,1
F9,1,2,1,1
HA,1,2,1,1
MQ,1,2,1,1


In [29]:
f_df.loc[
    (f_df['UNIQUE_CARRIER'] == 'UA') ,
    ['UNIQUE_CARRIER' , 'UNIQUE_CARRIER_ENTITY']
].drop_duplicates()

,UNIQUE_CARRIER,UNIQUE_CARRIER_ENTITY
2398,UA,0A875
2517,UA,10876
5067,UA,10874
5855,UA,10877


In [30]:
f_df.loc[
    (f_df['UNIQUE_CARRIER'] == 'UA') ,
    ['REGION' , 'UNIQUE_CARRIER_ENTITY']
].drop_duplicates().sort_values(by='UNIQUE_CARRIER_ENTITY')   

,REGION,UNIQUE_CARRIER_ENTITY
2398,D,0A875
5067,A,10874
2517,L,10876
5855,P,10877


In [31]:
multiple_entity_carriers = carrier_identifier_investigation[
    carrier_identifier_investigation["UNIQUE_CARRIER_ENTITY"] > 1
]

multiple_entity_carriers

,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
UNIQUE_CARRIER,,,,
3M,1,2,1,1
9E,1,2,1,1
AA,1,4,1,1
AS,1,2,1,1
B6,1,3,1,1
DL,1,4,1,1
F9,1,2,1,1
HA,1,2,1,1
MQ,1,2,1,1


In [32]:
f_df.loc[
    f_df["UNIQUE_CARRIER"] == 'DL',
    ['REGION' , 'UNIQUE_CARRIER_ENTITY']
].drop_duplicates()

,REGION,UNIQUE_CARRIER_ENTITY
2729,D,01260
3172,L,10260
5926,P,10262
5930,A,10261


In [33]:
carrier_region = f_df.loc[ 
    :,
    ['UNIQUE_CARRIER', 'UNIQUE_CARRIER_ENTITY','REGION']
].groupby("UNIQUE_CARRIER").agg(
    unique_entities = ('UNIQUE_CARRIER_ENTITY', 'nunique'),
    regions = ('REGION', 'nunique')
)
carrier_region

,unique_entities,regions
UNIQUE_CARRIER,,
07Q,1,1
1QQ,1,1
2E,1,1
2K,1,1
2NQ,1,1
...,...,...
YV,2,2
YX,1,1
Z0,1,1


In [34]:
multiple_carrier_region = carrier_region[
   carrier_region["regions"] > 1 
]
multiple_carrier_region

,unique_entities,regions
UNIQUE_CARRIER,,
3M,2,2
9E,2,2
AA,4,4
AS,2,2
B6,3,3
DL,4,4
F9,2,2
HA,2,2
MQ,2,2


In [35]:
carrier_region_mismatch = carrier_region[
    carrier_region["unique_entities"] != carrier_region["regions"]
]
carrier_region_mismatch

,unique_entities,regions
UNIQUE_CARRIER,,


In [36]:
regions_entity_map = f_df.loc[
    :,
    ['UNIQUE_CARRIER','REGION','UNIQUE_CARRIER_ENTITY']
].groupby(['UNIQUE_CARRIER','REGION']).nunique()

mismatched_regions_entity_map = regions_entity_map.loc[
    regions_entity_map['UNIQUE_CARRIER_ENTITY'] > 1
]
mismatched_regions_entity_map

,,UNIQUE_CARRIER_ENTITY
UNIQUE_CARRIER,REGION,


In [37]:
entity_to_region_map = f_df.loc[
    :,
    ['UNIQUE_CARRIER','REGION','UNIQUE_CARRIER_ENTITY']
].groupby(['UNIQUE_CARRIER','UNIQUE_CARRIER_ENTITY']).nunique()

entity_to_region_map_mismatch = entity_to_region_map.loc[
    entity_to_region_map['REGION'] > 1
    ]
    
entity_to_region_map_mismatch

,,REGION
UNIQUE_CARRIER,UNIQUE_CARRIER_ENTITY,


- Several carriers have multiple UNIQUE_CARRIER_ENTITY values. In the January 2024 scheduled-service data
- these entity values map one-to-one with carrier REGION
- This indicates that UNIQUE_CARRIER_ENTITY is a finer carrier-region reporting identifier rather than an airline-level identifier. Because the project unit of analysis is airline + year + month + origin + destination, this field appears too granular for identifying the airline itself. Longitudinal identifier stability still needs to be checked using multi-month data.

In [38]:
carrier_identifier_investigation [
    carrier_identifier_investigation['AIRLINE_ID'] > 1
    ]

,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
UNIQUE_CARRIER,,,,


In [39]:
carrier_identifier_investigation [
    carrier_identifier_investigation['CARRIER'] > 1
    ]

,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
UNIQUE_CARRIER,,,,


In [40]:
carrier_identifier_investigation [
    carrier_identifier_investigation['CARRIER_NAME'] > 1
    ]

,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
UNIQUE_CARRIER,,,,


In [41]:
carrier_identifier_investigation

,AIRLINE_ID,UNIQUE_CARRIER_ENTITY,CARRIER,CARRIER_NAME
UNIQUE_CARRIER,,,,
07Q,1,1,1,1
1QQ,1,1,1,1
2E,1,1,1,1
2K,1,1,1,1
2NQ,1,1,1,1
...,...,...,...,...
YV,1,2,1,1
YX,1,1,1,1
Z0,1,1,1,1


In [42]:
raw_dir = Path("../data/raw")

matching_files = list(
    raw_dir.glob("T_T100_SEGMENT_ALL_CARRIER_February*.zip")
)

if len(matching_files) != 1:
    raise ValueError(
        f"Expected exactly one T-100 Segment All Carriers ZIP, "
        f"found {len(matching_files)}"
    )

raw_path = matching_files[0]

In [43]:
feb_df = pd.read_csv(raw_path)


In [44]:
feb_df.shape

(42336, 50)

In [45]:
print("unique years:", feb_df['YEAR'].unique())
print("unique months:", feb_df['MONTH'].unique())


unique years: [2024]
unique months: [2]


In [46]:
required_columns = [
       "UNIQUE_CARRIER",
        "CARRIER_NAME",
        "ORIGIN",
        "DEST",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED",
]
set(required_columns).issubset(feb_df.columns)

True

In [47]:
print("Columns match:", df.columns.equals(feb_df.columns))

Columns match: True


In [48]:
raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2024.zip")

if not raw_path.is_file():
    raise FileNotFoundError(f"File not found: {raw_path}")

In [49]:
df_2024 = pd.read_csv(raw_path)


In [50]:
print("Shape of df_2024:", df_2024.shape)
print('Year: ', df_2024['YEAR'].unique())
print('Month: ', sorted(df_2024['MONTH'].unique()))


Shape of df_2024: (549731, 50)
Year:  [2024]
Month:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


In [51]:
from pathlib import Path

raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2025.zip")

print("1 - created")
print(raw_path)
print("2 - printed")
print(raw_path.resolve())
print("3 - resolved")
print(raw_path.exists())
print("4 - exists checked")

1 - created
..\data\raw\T_T100_SEGMENT_ALL_CARRIER_2025.zip
2 - printed
C:\Users\PC\airline-demand-capacity-planning\data\raw\T_T100_SEGMENT_ALL_CARRIER_2025.zip
3 - resolved
True
4 - exists checked


In [52]:
df_2025 = pd.read_csv(raw_path)
print("Shape of df_2025:", df_2025.shape)
print('Year: ', df_2025['YEAR'].unique())
print('Month: ', sorted(df_2025['MONTH'].unique()))

Shape of df_2025: (571097, 50)
Year:  [2025]
Month:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


In [53]:
df_2025.columns.equals(df_2024.columns)

True

In [54]:
raw_dir = Path("../data/raw")

matching_files = list(
    raw_dir.glob("T_T100_SEGMENT_ALL_CARRIER_2026.zip")
)

if len(matching_files) != 1:
    raise ValueError(
        f"Expected exactly one T-100 Segment All Carriers ZIP, "
        f"found {len(matching_files)}"
    )

raw_path = matching_files[0]

In [55]:
df_2026 = pd.read_csv(raw_path)
print("Shape of df_2026:", df_2026.shape)
print('Year: ', df_2026['YEAR'].unique())
print('Month: ', sorted(df_2026['MONTH'].unique()))

Shape of df_2026: (235688, 50)
Year:  [2026]
Month:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [56]:
df_2026.columns.equals(df_2025.columns)

True

In [57]:
df_historical = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)
print("Shape of df_historical:", df_historical.shape)

Shape of df_historical: (1356516, 50)


In [58]:
YEAR_MONTH = df_historical[['YEAR', 'MONTH']].drop_duplicates().sort_values(by=['YEAR', 'MONTH']).reset_index(drop=True)
YEAR_MONTH

,YEAR,MONTH
0,2024,1
1,2024,2
2,2024,3
3,2024,4
4,2024,5
5,2024,6
6,2024,7
7,2024,8
8,2024,9
9,2024,10


In [59]:
f_historical = df_historical.loc[df_historical['CLASS'] == 'F'].copy()
print("Shape of f_historical:", f_historical.shape)
print("Classes included: " , f_historical['CLASS'].unique())

Shape of f_historical: (1050113, 50)
Classes included:  <ArrowStringArray>
['F']
Length: 1, dtype: str


In [60]:
historical_carrier_identifier_investigation = f_historical[[
    'UNIQUE_CARRIER',
    'AIRLINE_ID',
    'CARRIER',
    'CARRIER_NAME',
    'UNIQUE_CARRIER_NAME',
]].groupby('UNIQUE_CARRIER').nunique()


In [61]:
mask = (historical_carrier_identifier_investigation > 1).any(axis=1)
historical_carrier_identifier_investigation[mask]

,AIRLINE_ID,CARRIER,CARRIER_NAME,UNIQUE_CARRIER_NAME
UNIQUE_CARRIER,,,,
9K,1,1,2,1
JJ,1,1,2,1
KG,1,1,2,1
LA,1,1,2,1
TI,1,2,1,1
TJ,1,2,1,1
TP,1,1,2,1
VD,1,2,1,1
W7,1,2,1,1


In [62]:
f_historical[f_historical['UNIQUE_CARRIER'] == '9K'][[
    'YEAR',
    'MONTH',
    'UNIQUE_CARRIER', 
    'CARRIER', 
    'CARRIER_NAME', 
    'UNIQUE_CARRIER_NAME', 
    'AIRLINE_ID']].drop_duplicates().sort_values(by=['YEAR', 'MONTH']).reset_index(drop=True)


,YEAR,MONTH,UNIQUE_CARRIER,CARRIER,CARRIER_NAME,UNIQUE_CARRIER_NAME,AIRLINE_ID
0,2024,1,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
1,2024,2,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
2,2024,3,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
3,2024,4,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
4,2024,5,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
5,2024,6,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
6,2024,7,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
7,2024,8,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
8,2024,9,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253
9,2024,10,9K,9K,Cape Air,"Hyannis Air Service, Inc. dba Cape Air",20253


In [63]:
f_historical[f_historical['UNIQUE_CARRIER'] == 'TI'][[
    'YEAR',
    'MONTH',
    'UNIQUE_CARRIER', 
    'CARRIER', 
    'CARRIER_NAME', 
    'UNIQUE_CARRIER_NAME', 
    'AIRLINE_ID']].drop_duplicates().sort_values(by=['YEAR', 'MONTH']).reset_index(drop=True)


,YEAR,MONTH,UNIQUE_CARRIER,CARRIER,CARRIER_NAME,UNIQUE_CARRIER_NAME,AIRLINE_ID
0,2024,1,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
1,2024,2,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
2,2024,3,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
3,2024,4,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
4,2024,5,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
5,2024,6,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
6,2024,7,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
7,2024,8,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
8,2024,9,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743
9,2024,10,TI,2LQ,Tropic Ocean Airways LLC,Tropic Ocean Airways LLC,21743


In [64]:
carrier_region = f_historical[
['UNIQUE_CARRIER', 'REGION', 'UNIQUE_CARRIER_ENTITY']
].groupby(['UNIQUE_CARRIER', 'REGION']).nunique()
carrier_region = carrier_region[carrier_region['UNIQUE_CARRIER_ENTITY'] > 1]
carrier_region

,,UNIQUE_CARRIER_ENTITY
UNIQUE_CARRIER,REGION,


In [65]:
carrier_region = f_historical[
['UNIQUE_CARRIER', 'REGION', 'UNIQUE_CARRIER_ENTITY']
].groupby(['UNIQUE_CARRIER', 'UNIQUE_CARRIER_ENTITY']).nunique()
carrier_region = carrier_region[carrier_region['REGION'] > 1]
carrier_region

,,REGION
UNIQUE_CARRIER,UNIQUE_CARRIER_ENTITY,


In [66]:
airline_id_check = f_historical[
    ['UNIQUE_CARRIER', 'AIRLINE_ID']
].groupby('AIRLINE_ID').agg(
    UNIQUE_CARRIER=('UNIQUE_CARRIER', 'nunique')
)

airline_id_check[airline_id_check['UNIQUE_CARRIER'] > 1]


,UNIQUE_CARRIER
AIRLINE_ID,


In [67]:
active_f_historical = f_historical[f_historical['PASSENGERS'] > 0].copy()
print("Shape of active_f_historical:", active_f_historical.shape)
print("minimum passengers:", active_f_historical['PASSENGERS'].min())


Shape of active_f_historical: (1011911, 50)
minimum passengers: 1.0


In [68]:
active_f_historical['UNIQUE_CARRIER'].nunique()

188

In [69]:
carrier_passenger_volume = active_f_historical.groupby('UNIQUE_CARRIER').agg(
    total_passengers=('PASSENGERS', 'sum')
).sort_values(by='total_passengers', ascending=False)
carrier_passenger_volume

,total_passengers
UNIQUE_CARRIER,
WN,414121175.0
AA,407907630.0
DL,400876174.0
UA,347375734.0
OO,106210884.0
...,...
WPT,609.0
VR,429.0
WRD,346.0


In [70]:
breadth_by_carrier = active_f_historical[
    ['UNIQUE_CARRIER', 
    'ORIGIN', 
    'DEST'
    ]
    ].drop_duplicates().groupby('UNIQUE_CARRIER').size().sort_values(ascending=False)
breadth_by_carrier.head(10)

UNIQUE_CARRIER
AA    4661
DL    4512
UA    4276
WN    4217
OO    2840
MQ    2438
YX    1886
B6    1593
OH    1569
G4    1500
dtype: int64

In [71]:
active_months_by_carrier = active_f_historical[
    ["UNIQUE_CARRIER","YEAR","MONTH"]
    ].drop_duplicates().groupby('UNIQUE_CARRIER').size().sort_values(ascending=False)
active_months_by_carrier.head(10)

UNIQUE_CARRIER
07Q    29
2E     29
2K     29
2O     29
3U     29
2OQ    29
6F     29
5V     29
5D     29
4Y     29
dtype: int64

In [72]:
active_months_by_carrier.tail(10)

UNIQUE_CARRIER
FK     9
2T     8
GQQ    7
E9     6
GF     5
SEB    3
Y9     2
AQB    1
WPT    1
VR     1
dtype: int64

In [73]:
active_months_by_carrier.value_counts().sort_index()

1       3
2       1
3       1
5       1
6       1
7       1
8       1
9       4
10      2
11      1
12      1
14      1
16      1
18      2
19      4
20      3
22      1
23      4
24      3
25      4
26      4
27     11
28     25
29    108
Name: count, dtype: int64

In [74]:
route_active_months = active_f_historical[
    ['UNIQUE_CARRIER', 
    'ORIGIN', 'DEST', 
    'YEAR', 'MONTH']
    ].drop_duplicates().groupby(['UNIQUE_CARRIER', 'ORIGIN', 'DEST']).size().sort_values(ascending=False)

In [75]:
print(route_active_months.head(10))
route_active_months.describe()

UNIQUE_CARRIER  ORIGIN  DEST
07Q             LAX     YVR     29
                FLL     YYZ     29
G4              LAS     GRR     29
                        GTF     29
                        ICT     29
OO              MFR     PHX     29
                MEI     IAH     29
                MDW     MSP     29
B6              EWR     FLL     29
                        CUN     29
dtype: int64


count    47278.000000
mean        11.150493
std         11.446310
min          1.000000
25%          1.000000
50%          5.000000
75%         24.000000
max         29.000000
dtype: float64

In [76]:
route_active_months.value_counts().sort_index()

1     15137
2      4553
3      2280
4      1632
5      1243
6       918
7       758
8       806
9       646
10      611
11      662
12      581
13      534
14      524
15      485
16      474
17      408
18      503
19      463
20      429
21      403
22      417
23      442
24      563
25      570
26      495
27      649
28     1223
29     8869
Name: count, dtype: int64

In [77]:
persistent_routes_29 = route_active_months[route_active_months == 29]
persistent_routes_29

UNIQUE_CARRIER  ORIGIN  DEST
07Q             LAX     YVR     29
                FLL     YYZ     29
G4              LAS     GRR     29
                        GTF     29
                        ICT     29
                                ..
                        MLI     29
                        MFR     29
                        MFE     29
                        MEM     29
                        LRD     29
Length: 8869, dtype: int64

In [78]:
persistent_routes_by_carrier = persistent_routes_29.groupby("UNIQUE_CARRIER").size().sort_values(ascending=False)
persistent_routes_by_carrier

UNIQUE_CARRIER
WN    1309
DL     969
UA     964
AA     906
OO     604
      ... 
KQ       2
SN       2
Q6       2
XL       2
4C       1
Length: 107, dtype: int64

In [79]:
carrier_route_stability = pd.concat([persistent_routes_by_carrier, breadth_by_carrier],axis=1)
carrier_route_stability.isna().sum()

0    81
1     0
dtype: int64

In [80]:
carrier_route_stability.columns = ["Persistent Routes", "Route Breadth"]
carrier_route_stability.head()

,Persistent Routes,Route Breadth
UNIQUE_CARRIER,,
WN,1309.0,4217
DL,969.0,4512
UA,964.0,4276
AA,906.0,4661
OO,604.0,2840


In [81]:
carrier_route_stability["Persistent Routes"] = carrier_route_stability["Persistent Routes"].fillna(0)
carrier_route_stability.isna().sum()

Persistent Routes    0
Route Breadth        0
dtype: int64

In [82]:
carrier_route_stability["Persistence Percentage"] = carrier_route_stability["Persistent Routes"] / carrier_route_stability["Route Breadth"] * 100
carrier_route_stability.sort_values(by="Persistence Percentage", ascending=False).head(10)

,Persistent Routes,Route Breadth,Persistence Percentage
UNIQUE_CARRIER,,,
2E,11.0,11,100.000000
NH,24.0,24,100.000000
AT,6.0,6,100.000000
CA,10.0,10,100.000000
KQ,2.0,2,100.000000
3U,2.0,2,100.000000
HY,2.0,2,100.000000
JU,2.0,2,100.000000
NEW,2.0,2,100.000000


In [83]:
carrier_route_stability["Route Breadth"].describe()

count     188.000000
mean      251.478723
std       740.146917
min         1.000000
25%         8.000000
50%        22.000000
75%        92.000000
max      4661.000000
Name: Route Breadth, dtype: float64

In [84]:
carrier_route_stability["Route Breadth"].quantile([0.85, 0.9, 0.95])

0.85     228.15
0.90     549.30
0.95    1464.30
Name: Route Breadth, dtype: float64

In [85]:
large_carrier_threshold = carrier_route_stability["Route Breadth"].quantile(0.95)
large_carriers = carrier_route_stability[carrier_route_stability["Route Breadth"] >= large_carrier_threshold]
large_carriers.sort_values(by="Persistence Percentage", ascending=False)

,Persistent Routes,Route Breadth,Persistence Percentage
UNIQUE_CARRIER,,,
WN,1309.0,4217,31.041024
G4,454.0,1500,30.266667
UA,964.0,4276,22.544434
DL,969.0,4512,21.476064
OO,604.0,2840,21.267606
B6,325.0,1593,20.401758
AA,906.0,4661,19.437889
YX,223.0,1886,11.823966
OH,182.0,1569,11.599745


In [86]:
carrier_passenger_volume.head(10)

,total_passengers
UNIQUE_CARRIER,
WN,414121175.0
AA,407907630.0
DL,400876174.0
UA,347375734.0
OO,106210884.0
B6,96307022.0
AS,90549125.0
NK,84015382.0
F9,81223287.0


In [87]:
large_carrier_comparison = large_carriers.join(carrier_passenger_volume, how="left")
large_carrier_comparison["total_passengers"] = large_carrier_comparison["total_passengers"]/1000000
large_carrier_comparison.rename(columns={"total_passengers": "Total Passengers (millions)"}, inplace=True)
large_carrier_comparison

,Persistent Routes,Route Breadth,Persistence Percentage,Total Passengers (millions)
UNIQUE_CARRIER,,,,
WN,1309.0,4217,31.041024,414.121175
DL,969.0,4512,21.476064,400.876174
UA,964.0,4276,22.544434,347.375734
AA,906.0,4661,19.437889,407.907630
OO,604.0,2840,21.267606,106.210884
G4,454.0,1500,30.266667,43.086228
B6,325.0,1593,20.401758,96.307022
YX,223.0,1886,11.823966,49.263621
MQ,210.0,2438,8.613618,45.135796


In [88]:
median_route_active_months = route_active_months[large_carriers.index].groupby("UNIQUE_CARRIER").median().sort_values(ascending=False)
median_route_active_months

UNIQUE_CARRIER
G4    18.0
OO    11.0
WN     6.0
YX     5.0
B6     3.0
AA     3.0
OH     2.0
MQ     2.0
DL     2.0
UA     2.0
dtype: float64

In [89]:
median_route_active_months.name = "Median Route Active Months"
large_carrier_comparison = large_carrier_comparison.join(median_route_active_months, how="left")
large_carrier_comparison

,Persistent Routes,Route Breadth,Persistence Percentage,Total Passengers (millions),Median Route Active Months
UNIQUE_CARRIER,,,,,
WN,1309.0,4217,31.041024,414.121175,6.0
DL,969.0,4512,21.476064,400.876174,2.0
UA,964.0,4276,22.544434,347.375734,2.0
AA,906.0,4661,19.437889,407.907630,3.0
OO,604.0,2840,21.267606,106.210884,11.0
G4,454.0,1500,30.266667,43.086228,18.0
B6,325.0,1593,20.401758,96.307022,3.0
YX,223.0,1886,11.823966,49.263621,5.0
MQ,210.0,2438,8.613618,45.135796,2.0


In [90]:
high_recurrence_routes = route_active_months[route_active_months >= 24]
len(high_recurrence_routes)

12369

In [91]:
high_recurrence_routes_by_carrier = high_recurrence_routes.groupby("UNIQUE_CARRIER").size().sort_values(ascending=False)
high_recurrence_routes_by_carrier.head(10)

UNIQUE_CARRIER
WN    1515
UA    1126
AA    1119
DL    1065
OO     886
G4     643
F9     442
NK     376
MQ     356
YX     354
dtype: int64

In [92]:
high_recurrence_routes_by_carrier.name = "High Recurrence Routes"
large_carrier_comparison = large_carrier_comparison.join(high_recurrence_routes_by_carrier, how="left")
large_carrier_comparison

,Persistent Routes,Route Breadth,Persistence Percentage,Total Passengers (millions),Median Route Active Months,High Recurrence Routes
UNIQUE_CARRIER,,,,,,
WN,1309.0,4217,31.041024,414.121175,6.0,1515
DL,969.0,4512,21.476064,400.876174,2.0,1065
UA,964.0,4276,22.544434,347.375734,2.0,1126
AA,906.0,4661,19.437889,407.907630,3.0,1119
OO,604.0,2840,21.267606,106.210884,11.0,886
G4,454.0,1500,30.266667,43.086228,18.0,643
B6,325.0,1593,20.401758,96.307022,3.0,351
YX,223.0,1886,11.823966,49.263621,5.0,354
MQ,210.0,2438,8.613618,45.135796,2.0,356


In [93]:
large_carrier_comparison["High Recurrence Percentage"] = large_carrier_comparison["High Recurrence Routes"] / large_carrier_comparison["Route Breadth"] * 100
large_carrier_comparison[["Route Breadth", "High Recurrence Percentage", "High Recurrence Routes"]].sort_values(by="High Recurrence Percentage", ascending=False)

,Route Breadth,High Recurrence Percentage,High Recurrence Routes
UNIQUE_CARRIER,,,
G4,1500,42.866667,643
WN,4217,35.926014,1515
OO,2840,31.197183,886
UA,4276,26.333022,1126
AA,4661,24.007724,1119
DL,4512,23.603723,1065
B6,1593,22.033898,351
YX,1886,18.769883,354
OH,1569,18.291906,287


In [94]:
large_carrier_names = active_f_historical[
    active_f_historical
    ["UNIQUE_CARRIER"].isin(large_carrier_comparison.index)][
        ["UNIQUE_CARRIER","UNIQUE_CARRIER_NAME"]
        ].drop_duplicates().sort_values(by="UNIQUE_CARRIER_NAME")

large_carrier_names.set_index("UNIQUE_CARRIER", inplace=True)
large_carrier_names

,UNIQUE_CARRIER_NAME
UNIQUE_CARRIER,
G4,Allegiant Air
AA,American Airlines Inc.
DL,Delta Air Lines Inc.
MQ,Envoy Air
B6,JetBlue Airways
OH,PSA Airlines Inc.
YX,Republic Airline
OO,SkyWest Airlines Inc.
WN,Southwest Airlines Co.


In [95]:
large_carrier_comparison = large_carrier_comparison.join(large_carrier_names, how="left")
large_carrier_comparison

,Persistent Routes,Route Breadth,Persistence Percentage,Total Passengers (millions),Median Route Active Months,High Recurrence Routes,High Recurrence Percentage,UNIQUE_CARRIER_NAME
UNIQUE_CARRIER,,,,,,,,
WN,1309.0,4217,31.041024,414.121175,6.0,1515,35.926014,Southwest Airlines Co.
DL,969.0,4512,21.476064,400.876174,2.0,1065,23.603723,Delta Air Lines Inc.
UA,964.0,4276,22.544434,347.375734,2.0,1126,26.333022,United Air Lines Inc.
AA,906.0,4661,19.437889,407.907630,3.0,1119,24.007724,American Airlines Inc.
OO,604.0,2840,21.267606,106.210884,11.0,886,31.197183,SkyWest Airlines Inc.
G4,454.0,1500,30.266667,43.086228,18.0,643,42.866667,Allegiant Air
B6,325.0,1593,20.401758,96.307022,3.0,351,22.033898,JetBlue Airways
YX,223.0,1886,11.823966,49.263621,5.0,354,18.769883,Republic Airline
MQ,210.0,2438,8.613618,45.135796,2.0,356,14.602133,Envoy Air


In [96]:
large_carrier_comparison[
    [
    "UNIQUE_CARRIER_NAME",
    "Route Breadth",
    "Total Passengers (millions)",
    "High Recurrence Routes",
    "High Recurrence Percentage",
    "Median Route Active Months"]
].sort_values(by="High Recurrence Routes", ascending=False)

,UNIQUE_CARRIER_NAME,Route Breadth,Total Passengers (millions),High Recurrence Routes,High Recurrence Percentage,Median Route Active Months
UNIQUE_CARRIER,,,,,,
WN,Southwest Airlines Co.,4217,414.121175,1515,35.926014,6.0
UA,United Air Lines Inc.,4276,347.375734,1126,26.333022,2.0
AA,American Airlines Inc.,4661,407.907630,1119,24.007724,3.0
DL,Delta Air Lines Inc.,4512,400.876174,1065,23.603723,2.0
OO,SkyWest Airlines Inc.,2840,106.210884,886,31.197183,11.0
G4,Allegiant Air,1500,43.086228,643,42.866667,18.0
MQ,Envoy Air,2438,45.135796,356,14.602133,2.0
YX,Republic Airline,1886,49.263621,354,18.769883,5.0
B6,JetBlue Airways,1593,96.307022,351,22.033898,3.0


In [97]:
wn_route_months = active_f_historical[
    active_f_historical["UNIQUE_CARRIER"] == 'WN'
    ][["ORIGIN", "DEST", "YEAR", "MONTH"]
    ].drop_duplicates().sort_values(by=["YEAR", "MONTH"]).reset_index(drop=True)

wn_route_months.head(10)

,ORIGIN,DEST,YEAR,MONTH
0,IAH,SLC,2024,1
1,AMA,BWI,2024,1
2,TUL,BNA,2024,1
3,PNS,PIT,2024,1
4,ORF,GRR,2024,1
5,STL,MAF,2024,1
6,MKE,RDU,2024,1
7,MSP,SJC,2024,1
8,OAK,SJC,2024,1
9,TPA,MCO,2024,1


In [98]:
wn_route_months["YEAR MONTH"] = wn_route_months["YEAR"].astype(str) + "-" + wn_route_months["MONTH"].astype(str).str.zfill(2)
wn_route_months.head()

,ORIGIN,DEST,YEAR,MONTH,YEAR MONTH
0,IAH,SLC,2024,1,2024-01
1,AMA,BWI,2024,1,2024-01
2,TUL,BNA,2024,1,2024-01
3,PNS,PIT,2024,1,2024-01
4,ORF,GRR,2024,1,2024-01


In [99]:
wn_route_months["YEAR MONTH"] = (
    pd.to_datetime(wn_route_months["YEAR MONTH"])
    .dt.to_period("M")
)
wn_route_months.head()

,ORIGIN,DEST,YEAR,MONTH,YEAR MONTH
0,IAH,SLC,2024,1,2024-01
1,AMA,BWI,2024,1,2024-01
2,TUL,BNA,2024,1,2024-01
3,PNS,PIT,2024,1,2024-01
4,ORF,GRR,2024,1,2024-01


In [100]:
wn_route_active = route_active_months.loc["WN"]

wn_route_active = wn_route_active[
    (wn_route_active >= 24) & (wn_route_active < 29)]

wn_route_active.head(10)

ORIGIN  DEST
BUF     FLL     28
BNA     CUN     28
CUN     BNA     28
COS     HOU     28
FLL     BUF     28
DEN     RSW     28
HOU     VPS     28
        TUS     28
        SAV     28
        LBB     28
dtype: int64

In [101]:
hou_to_tus = wn_route_months[
    (wn_route_months['ORIGIN'] == 'HOU') & (wn_route_months['DEST'] == 'TUS')
    ].sort_values(by="YEAR MONTH").reset_index(drop=True)

In [102]:
hou_to_tus["YEAR MONTH"].astype(int).diff()

0     NaN
1     1.0
2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
7     1.0
8     1.0
9     1.0
10    1.0
11    1.0
12    1.0
13    1.0
14    1.0
15    1.0
16    1.0
17    1.0
18    1.0
19    1.0
20    1.0
21    1.0
22    1.0
23    1.0
24    1.0
25    1.0
26    1.0
27    1.0
Name: YEAR MONTH, dtype: float64

In [103]:
wn_route_months = wn_route_months.sort_values(by=["ORIGIN","DEST","YEAR MONTH"]).reset_index(drop=True)
wn_route_months.head(10)

,ORIGIN,DEST,YEAR,MONTH,YEAR MONTH
0,ABI,DAL,2025,10,2025-10
1,ABQ,AFW,2026,2,2026-02
2,ABQ,AMA,2024,8,2024-08
3,ABQ,AMA,2025,4,2025-04
4,ABQ,ATL,2024,7,2024-07
5,ABQ,AUS,2024,1,2024-01
6,ABQ,AUS,2024,2,2024-02
7,ABQ,AUS,2024,3,2024-03
8,ABQ,AUS,2024,4,2024-04
9,ABQ,AUS,2024,5,2024-05


In [104]:
wn_route_months["MONTH NUMBER"] = (
    wn_route_months["YEAR MONTH"].astype("int64")
)

In [105]:
wn_route_months["MONTH GAP"] = (
    wn_route_months
    .groupby(["ORIGIN", "DEST"])["MONTH NUMBER"]
    .diff()
)


In [106]:
wn_route_months

,ORIGIN,DEST,YEAR,MONTH,YEAR MONTH,MONTH NUMBER,MONTH GAP
0,ABI,DAL,2025,10,2025-10,669,NaN
1,ABQ,AFW,2026,2,2026-02,673,NaN
2,ABQ,AMA,2024,8,2024-08,655,NaN
3,ABQ,AMA,2025,4,2025-04,663,8.0
4,ABQ,ATL,2024,7,2024-07,654,NaN
...,...,...,...,...,...,...,...
55252,VPS,STL,2025,10,2025-10,669,1.0
55253,VPS,STL,2025,11,2025-11,670,1.0
55254,VPS,STL,2026,3,2026-03,674,4.0
55255,VPS,STL,2026,4,2026-04,675,1.0


In [107]:
wn_routes_with_internal_gaps = wn_route_months[wn_route_months["MONTH GAP"] > 1][[
    "ORIGIN",
    "DEST"
]].drop_duplicates().sort_values(by=["ORIGIN", "DEST"]).reset_index(drop=True)
len(wn_routes_with_internal_gaps)

1400

In [108]:
wn_high_recurrence = route_active_months.loc["WN"]
wn_high_recurrence = wn_high_recurrence[
    (wn_high_recurrence >= 24)
    ]
len(wn_high_recurrence)

1515

In [109]:
wn_high_recurrence_df = wn_high_recurrence.reset_index(
    name="ACTIVE_MONTHS"
)
len(wn_high_recurrence)

1515

In [110]:
wn_high_recurrence_with_gaps = wn_high_recurrence_df.merge(
    wn_routes_with_internal_gaps,
    on=["ORIGIN", "DEST"],
    how="inner"
)
len(wn_high_recurrence_with_gaps)

132

In [111]:
wn_internal_gap_percentage = len(wn_high_recurrence_with_gaps) / len(wn_high_recurrence_df) * 100
wn_internal_gap_percentage

8.712871287128712

In [112]:
non_full_high_recurrence = (
    len(wn_high_recurrence_df)
    - persistent_routes_by_carrier.loc["WN"]
)
non_full_high_recurrence

np.int64(206)

In [113]:
wn_high_recurrence_with_gaps.groupby("ACTIVE_MONTHS").size()

ACTIVE_MONTHS
24    19
25    45
26    17
27    30
28    21
dtype: int64

In [114]:
wn_gap_events = wn_route_months[
    wn_route_months["MONTH GAP"] > 1
][
    ["ORIGIN", "DEST", "YEAR MONTH", "MONTH GAP"]
]
len(wn_gap_events)

2796

In [115]:
wn_high_recurrence_gap_events = wn_high_recurrence_with_gaps.merge(
    wn_gap_events
    ,how="inner"
    , on=["ORIGIN", "DEST"]
    )
wn_high_recurrence_gap_events.head(10)

,ORIGIN,DEST,ACTIVE_MONTHS,YEAR MONTH,MONTH GAP
0,BUF,FLL,28,2025-10,2.0
1,BNA,CUN,28,2024-10,2.0
2,CUN,BNA,28,2024-10,2.0
3,COS,HOU,28,2026-03,2.0
4,FLL,BUF,28,2025-10,2.0
5,DEN,RSW,28,2025-10,2.0
6,HOU,VPS,28,2026-03,2.0
7,HOU,SAV,28,2026-03,2.0
8,HOU,LBB,28,2026-03,2.0
9,HOU,COS,28,2026-03,2.0


In [116]:
wn_high_recurrence_gap_events["MONTH GAP"].value_counts().sort_index()

MONTH GAP
2.0    139
3.0     73
4.0     14
5.0      7
6.0      1
Name: count, dtype: int64

In [117]:
(wn_high_recurrence_gap_events["MONTH GAP"].value_counts(normalize=True) * 100).sort_index()

MONTH GAP
2.0    59.401709
3.0    31.196581
4.0     5.982906
5.0     2.991453
6.0     0.427350
Name: proportion, dtype: float64

In [118]:
wn_high_recurrence_gap_events.groupby(
    ["ORIGIN", "DEST"]
).size().value_counts(normalize=True).sort_index() * 100

1    37.121212
2    53.787879
3     4.545455
4     3.787879
5     0.757576
Name: proportion, dtype: float64

In [119]:
wn_high_recurrence_gap_events["MISSING MONTHS"] = wn_high_recurrence_gap_events["MONTH GAP"] - 1
wn_high_recurrence_gap_events.head(10)

,ORIGIN,DEST,ACTIVE_MONTHS,YEAR MONTH,MONTH GAP,MISSING MONTHS
0,BUF,FLL,28,2025-10,2.0,1.0
1,BNA,CUN,28,2024-10,2.0,1.0
2,CUN,BNA,28,2024-10,2.0,1.0
3,COS,HOU,28,2026-03,2.0,1.0
4,FLL,BUF,28,2025-10,2.0,1.0
5,DEN,RSW,28,2025-10,2.0,1.0
6,HOU,VPS,28,2026-03,2.0,1.0
7,HOU,SAV,28,2026-03,2.0,1.0
8,HOU,LBB,28,2026-03,2.0,1.0
9,HOU,COS,28,2026-03,2.0,1.0


In [120]:
wn_high_recurrence_gap_events.groupby(["ORIGIN", "DEST"])['MISSING MONTHS'].sum().sort_values(ascending=False)

ORIGIN  DEST
ICT     PHX     5.0
MCO     DTW     5.0
MBJ     MDW     5.0
DTW     MCO     5.0
PVD     FLL     5.0
               ... 
SJC     OGG     1.0
SMF     BNA     1.0
TPA     MHT     1.0
        PHL     1.0
VPS     HOU     1.0
Name: MISSING MONTHS, Length: 132, dtype: float64

In [121]:
wn_high_recurrence_gap_events["TOTAL MISSING MONTHS"] = 29 - wn_high_recurrence_gap_events["ACTIVE_MONTHS"]
wn_high_recurrence_gap_events.head(10)

,ORIGIN,DEST,ACTIVE_MONTHS,YEAR MONTH,MONTH GAP,MISSING MONTHS,TOTAL MISSING MONTHS
0,BUF,FLL,28,2025-10,2.0,1.0,1
1,BNA,CUN,28,2024-10,2.0,1.0,1
2,CUN,BNA,28,2024-10,2.0,1.0,1
3,COS,HOU,28,2026-03,2.0,1.0,1
4,FLL,BUF,28,2025-10,2.0,1.0,1
5,DEN,RSW,28,2025-10,2.0,1.0,1
6,HOU,VPS,28,2026-03,2.0,1.0,1
7,HOU,SAV,28,2026-03,2.0,1.0,1
8,HOU,LBB,28,2026-03,2.0,1.0,1
9,HOU,COS,28,2026-03,2.0,1.0,1


In [122]:
wn_high_recurrence_gap_summary = wn_high_recurrence_gap_events.groupby(["ORIGIN", "DEST"]).agg(
    ACTIVE_MONTHS = ("ACTIVE_MONTHS", "first"),
    TOTAL_MISSING_MONTHS = ("TOTAL MISSING MONTHS", "first"),
    INTERNAL_MISSING_MONTHS = ("MISSING MONTHS", "sum")
)
wn_high_recurrence_gap_summary.head(10)

ACTIVE_MONTHS  TOTAL_MISSING_MONTHS  INTERNAL_MISSING_MONTHS
ORIGIN DEST                                                              
ALB    DEN              25                     4                      4.0
AUS    ORD              26                     3                      3.0
BHM    BNA              25                     4                      4.0
BNA    CUN              28                     1                      1.0
       SJC              25                     4                      2.0
       SJU              25                     4                      4.0
       SMF              26                     3                      1.0
BUF    FLL              28                     1                      1.0
BUR    MDW              25                     4                      4.0
BWI    LIR              25                     4                      4.0

In [123]:
wn_high_recurrence_gap_summary['EDGE_MISSING_MONTHS'] = wn_high_recurrence_gap_summary['TOTAL_MISSING_MONTHS'] - wn_high_recurrence_gap_summary['INTERNAL_MISSING_MONTHS']
wn_high_recurrence_gap_summary.head(10)

ACTIVE_MONTHS  TOTAL_MISSING_MONTHS  INTERNAL_MISSING_MONTHS  \
ORIGIN DEST                                                                 
ALB    DEN              25                     4                      4.0   
AUS    ORD              26                     3                      3.0   
BHM    BNA              25                     4                      4.0   
BNA    CUN              28                     1                      1.0   
       SJC              25                     4                      2.0   
       SJU              25                     4                      4.0   
       SMF              26                     3                      1.0   
BUF    FLL              28                     1                      1.0   
BUR    MDW              25                     4                      4.0   
BWI    LIR              25                     4                      4.0   

             EDGE_MISSING_MONTHS  
ORIGIN DEST                       
ALB    DEN                   0.0  
AUS    ORD                   0.0  
BHM    BNA                   0.0  
BNA    CUN                   0.0  
       SJC                   2.0  
       SJU                   0.0  
       SMF                   2.0  
BUF    FLL                   0.0  
BUR    MDW                   0.0  
BWI    LIR                   0.0

In [124]:
wn_high_recurrence_gap_summary["EDGE_MISSING_MONTHS"].value_counts().sort_index()

EDGE_MISSING_MONTHS
0.0    106
1.0      8
2.0     16
3.0      1
4.0      1
Name: count, dtype: int64

In [125]:
# 1. Active all 29 months
wn_full_29_count = (
    wn_high_recurrence_df["ACTIVE_MONTHS"] == 29
).sum()


# 2. Active 24-28 months, but with no internal gaps
wn_continuous_nonfull_count = (
    len(wn_high_recurrence_df)
    - wn_full_29_count
    - len(wn_high_recurrence_with_gaps)
)


# 3. Internal gaps, but no edge missing months
wn_internal_gap_only = (
    wn_high_recurrence_gap_summary["EDGE_MISSING_MONTHS"] == 0
).sum()


# 4. Both internal gaps and edge missing months
wn_both_gap_count = (
    wn_high_recurrence_gap_summary["EDGE_MISSING_MONTHS"] > 0
).sum()

In [126]:
print("full 29 months:", wn_full_29_count / wn_high_recurrence_df.shape[0] * 100)
print("continuous nonfull:", wn_continuous_nonfull_count / wn_high_recurrence_df.shape[0] * 100)
print("internal gaps only:", wn_internal_gap_only / wn_high_recurrence_df.shape[0] * 100)
print("both internal and edge gaps:", wn_both_gap_count / wn_high_recurrence_df.shape[0] * 100)

full 29 months: 86.4026402640264
continuous nonfull: 4.884488448844884
internal gaps only: 6.996699669966996
both internal and edge gaps: 1.7161716171617163


In [127]:
wn_continuous_count = (wn_full_29_count + wn_continuous_nonfull_count)
print("Continuous count (full 29 + continuous nonfull):", wn_continuous_count)
wn_continuous_percentage = (wn_full_29_count + wn_continuous_nonfull_count)/ wn_high_recurrence_df.shape[0] * 100
print("Continuous percentage (full 29 + continuous nonfull):", wn_continuous_percentage)

Continuous count (full 29 + continuous nonfull): 1383
Continuous percentage (full 29 + continuous nonfull): 91.2871287128713


In [128]:
large_carrier_comparison[
    ["Route Breadth",
     "Total Passengers (millions)",
     "Persistent Routes",
     "Persistence Percentage",
     "High Recurrence Routes",
     "High Recurrence Percentage",
     "Median Route Active Months"]          
].sort_values(by="High Recurrence Routes", ascending=False)

,Route Breadth,Total Passengers (millions),Persistent Routes,Persistence Percentage,High Recurrence Routes,High Recurrence Percentage,Median Route Active Months
UNIQUE_CARRIER,,,,,,,
WN,4217,414.121175,1309.0,31.041024,1515,35.926014,6.0
UA,4276,347.375734,964.0,22.544434,1126,26.333022,2.0
AA,4661,407.907630,906.0,19.437889,1119,24.007724,3.0
DL,4512,400.876174,969.0,21.476064,1065,23.603723,2.0
OO,2840,106.210884,604.0,21.267606,886,31.197183,11.0
G4,1500,43.086228,454.0,30.266667,643,42.866667,18.0
MQ,2438,45.135796,210.0,8.613618,356,14.602133,2.0
YX,1886,49.263621,223.0,11.823966,354,18.769883,5.0
B6,1593,96.307022,325.0,20.401758,351,22.033898,3.0


### Carrier Selection Conclusion

- WN (Southwest Airlines Co.) was selected as the project carrier.
- WN operated 4,217 observed directional routes during the historical period.
- It carried approximately 414.12 million passengers across the active scheduled-passenger data.
- WN had 1,515 high-recurrence routes active in at least 24 of the 29 historical months.
- Of these, 1,309 routes were active during all 29 months.
- Approximately 91.29% of WN's high-recurrence routes had continuous monthly histories with no internal gaps.
- Although G4 had a higher high-recurrence percentage, its network scale and passenger volume were substantially smaller.
- WN therefore provides the strongest overall balance of route scale, passenger demand volume, and historical route stability for this project.

In [129]:
wn_threshold_counts = (
    wn_high_recurrence_df["ACTIVE_MONTHS"]
    .value_counts()
    .sort_index(ascending=False)
    .cumsum()
    .sort_index()
)

wn_threshold_counts

ACTIVE_MONTHS
24    1515
25    1475
26    1418
27    1396
28    1338
29    1309
Name: count, dtype: int64

### Route Recurrence Threshold Note

The 24–29 month recurrence thresholds are used only for exploratory analysis of historical route stability.

They must not be used directly as a route-eligibility rule during historical backtesting because the recurrence counts use information from the full Jan 2024–May 2026 period. Using future months to determine whether a route was eligible at an earlier forecast date would introduce look-ahead leakage.

The final modeling route-eligibility rule must therefore be based only on information available as of each forecast origin.

In [130]:
wn_route_first_active = wn_route_months.groupby(["ORIGIN", "DEST"])["YEAR MONTH"].agg('min').reset_index(name="FIRST_ACTIVE_MONTH")
wn_route_first_active.head(10)

,ORIGIN,DEST,FIRST_ACTIVE_MONTH
0,ABI,DAL,2025-10
1,ABQ,AFW,2026-02
2,ABQ,AMA,2024-08
3,ABQ,ATL,2024-07
4,ABQ,AUS,2024-01
5,ABQ,BHM,2026-02
6,ABQ,BNA,2024-07
7,ABQ,BUR,2024-01
8,ABQ,BWI,2024-01
9,ABQ,CMH,2025-10


In [131]:
wn_route_first_active["FIRST_ACTIVE_MONTH"].value_counts().sort_index()

FIRST_ACTIVE_MONTH
2024-01    1831
2024-02      74
2024-03     229
2024-04      89
2024-05     182
2024-06     239
2024-07     152
2024-08      95
2024-09      42
2024-10      40
2024-11      37
2024-12      55
2025-01      35
2025-02      45
2025-03      76
2025-04      65
2025-05      69
2025-06     147
2025-07     168
2025-08      82
2025-09      43
2025-10      35
2025-11      70
2025-12      42
2026-01      27
2026-02      59
2026-03     106
2026-04      44
2026-05      39
Freq: M, Name: count, dtype: int64

In [132]:
First_observed_2024_01 = wn_route_first_active[
    wn_route_first_active["FIRST_ACTIVE_MONTH"] == pd.Period("2024-01")
].shape[0]
First_observed_after_2024_01 = wn_route_first_active[
    wn_route_first_active["FIRST_ACTIVE_MONTH"] > pd.Period("2024-01")
].shape[0]
print("First observed in 2024-01: ", First_observed_2024_01)
print("First observed after 2024-01: ", First_observed_after_2024_01)

First observed in 2024-01:  1831
First observed after 2024-01:  2386


In [133]:
First_observed_2024_01_percentage = First_observed_2024_01 / wn_route_first_active.shape[0] * 100
First_observed_after_2024_01_percentage = First_observed_after_2024_01 / wn_route_first_active.shape[0] * 100
print("First observed in 2024-01 percentage: ", First_observed_2024_01_percentage)
print("First observed after 2024-01 percentage: ", First_observed_after_2024_01_percentage)

First observed in 2024-01 percentage:  43.41949253023476
First observed after 2024-01 percentage:  56.58050746976524


In [134]:
wn_high_recurrence_history = wn_high_recurrence_df.merge(
    wn_route_first_active,
    on=["ORIGIN", "DEST"],
    how="left"
)
wn_high_recurrence_history.head(10)

,ORIGIN,DEST,ACTIVE_MONTHS,FIRST_ACTIVE_MONTH
0,TUS,LAS,29,2024-01
1,TUS,DEN,29,2024-01
2,VPS,DAL,29,2024-01
3,VPS,BNA,29,2024-01
4,TUS,SAN,29,2024-01
5,TUS,MDW,29,2024-01
6,TUS,LAX,29,2024-01
7,TUL,LAS,29,2024-01
8,TUL,HOU,29,2024-01
9,TUL,DEN,29,2024-01


In [135]:
print("Shape:", wn_high_recurrence_history.shape)

print(
    "Missing FIRST_ACTIVE_MONTH:",
    wn_high_recurrence_history["FIRST_ACTIVE_MONTH"].isna().sum()
)

Shape: (1515, 4)
Missing FIRST_ACTIVE_MONTH: 0


In [136]:
wn_high_recurrence_history["FIRST_ACTIVE_MONTH"].value_counts().sort_index()

FIRST_ACTIVE_MONTH
2024-01    1466
2024-03      20
2024-04       6
2024-05       2
2024-06      21
Freq: M, Name: count, dtype: int64

In [137]:
first_observed_2024_01_count = wn_high_recurrence_history[
    wn_high_recurrence_history["FIRST_ACTIVE_MONTH"] == pd.Period("2024-01")
].shape[0]

first_observed_after_2024_01_count = wn_high_recurrence_history[
    wn_high_recurrence_history["FIRST_ACTIVE_MONTH"] > pd.Period("2024-01")
].shape[0]

print("First observed in 2024-01 count: ", first_observed_2024_01_count)
print("First observed after 2024-01 count: ", first_observed_after_2024_01_count)

First observed in 2024-01 count:  1466
First observed after 2024-01 count:  49


In [138]:
HISTORICAL_ACTIVE_MONTHS = wn_route_months[
    wn_route_months["YEAR MONTH"] < pd.Period("2025-01")
].groupby(["ORIGIN", "DEST"])["YEAR MONTH"].agg('count').reset_index(name="HISTORICAL_ACTIVE_MONTHS")
HISTORICAL_ACTIVE_MONTHS.head(10)

,ORIGIN,DEST,HISTORICAL_ACTIVE_MONTHS
0,ABQ,AMA,1
1,ABQ,ATL,1
2,ABQ,AUS,12
3,ABQ,BNA,3
4,ABQ,BUR,12
5,ABQ,BWI,12
6,ABQ,COS,4
7,ABQ,CRP,1
8,ABQ,DAL,12
9,ABQ,DCA,1


In [139]:
HISTORICAL_ACTIVE_MONTHS["HISTORICAL_ACTIVE_MONTHS"].value_counts().sort_index()

HISTORICAL_ACTIVE_MONTHS
1      779
2      222
3      105
4       87
5       55
6       57
7       75
8       70
9       41
10      80
11      61
12    1433
Name: count, dtype: int64

In [140]:
total_routes = HISTORICAL_ACTIVE_MONTHS.shape[0]
eligible_12 = HISTORICAL_ACTIVE_MONTHS[HISTORICAL_ACTIVE_MONTHS["HISTORICAL_ACTIVE_MONTHS"] >= 12].shape[0]
eligible_12_percentage = (eligible_12 / total_routes) * 100

print("Total routes: ", total_routes)
print("Routes with 12+ prior active months: ", eligible_12)
print("Percentage: ", eligible_12_percentage)

Total routes:  3065
Routes with 12+ prior active months:  1433
Percentage:  46.7536704730832


In [141]:
forecast_origin = pd.Period("2025-07")
HISTORICAL_ACTIVE_MONTHS_2025_07 = wn_route_months[
    wn_route_months["YEAR MONTH"] < forecast_origin
].groupby(["ORIGIN", "DEST"])["YEAR MONTH"].agg('count').reset_index(name="HISTORICAL_ACTIVE_MONTHS_2025_07")

HISTORICAL_ACTIVE_MONTHS_2025_07.head(10)

,ORIGIN,DEST,HISTORICAL_ACTIVE_MONTHS_2025_07
0,ABQ,AMA,2
1,ABQ,ATL,1
2,ABQ,AUS,18
3,ABQ,BNA,6
4,ABQ,BUR,18
5,ABQ,BWI,18
6,ABQ,COS,5
7,ABQ,CRP,1
8,ABQ,DAL,18
9,ABQ,DCA,1


In [142]:
total_routes_2025_07 = HISTORICAL_ACTIVE_MONTHS_2025_07.shape[0]

eligible_12_2025_07 = HISTORICAL_ACTIVE_MONTHS_2025_07[
    HISTORICAL_ACTIVE_MONTHS_2025_07["HISTORICAL_ACTIVE_MONTHS_2025_07"] >= 12
].shape[0]

eligible_12_2025_07_percentage = (
    eligible_12_2025_07 / total_routes_2025_07
) * 100

print("Total routes:", total_routes_2025_07)
print("Routes with 12+ historical active months:", eligible_12_2025_07)
print("Percentage:", eligible_12_2025_07_percentage)

Total routes: 3502
Routes with 12+ historical active months: 1675
Percentage: 47.82981153626499


In [143]:
forecast_origin = pd.Period("2026-01")
HISTORICAL_ACTIVE_MONTHS_2026_01 = wn_route_months[
    wn_route_months["YEAR MONTH"] < forecast_origin
].groupby(["ORIGIN", "DEST"])["YEAR MONTH"].agg('count').reset_index(name="HISTORICAL_ACTIVE_MONTHS_2026_01")

HISTORICAL_ACTIVE_MONTHS_2026_01.head(10)

,ORIGIN,DEST,HISTORICAL_ACTIVE_MONTHS_2026_01
0,ABI,DAL,1
1,ABQ,AMA,2
2,ABQ,ATL,1
3,ABQ,AUS,24
4,ABQ,BNA,12
5,ABQ,BUR,24
6,ABQ,BWI,24
7,ABQ,CMH,1
8,ABQ,COS,6
9,ABQ,CRP,1


In [144]:
total_routes_2026_01 = HISTORICAL_ACTIVE_MONTHS_2026_01.shape[0]
eligible_12_2026_01 = HISTORICAL_ACTIVE_MONTHS_2026_01[HISTORICAL_ACTIVE_MONTHS_2026_01["HISTORICAL_ACTIVE_MONTHS_2026_01"] >= 12].shape[0]
eligible_12_2026_01_percentage = (eligible_12_2026_01 / total_routes_2026_01) * 100

print("Total routes: ", total_routes_2026_01)
print("Routes with 12+ prior active months: ", eligible_12_2026_01)
print("Percentage: ", eligible_12_2026_01_percentage)

Total routes:  3942
Routes with 12+ prior active months:  1768
Percentage:  44.85032978183663


In [145]:
HISTORICAL_ACTIVE_MONTHS_6_MONTHS = HISTORICAL_ACTIVE_MONTHS[
    HISTORICAL_ACTIVE_MONTHS["HISTORICAL_ACTIVE_MONTHS"] >= 6
]
HISTORICAL_ACTIVE_MONTHS_6_MONTHS.head(10)

,ORIGIN,DEST,HISTORICAL_ACTIVE_MONTHS
2,ABQ,AUS,12
4,ABQ,BUR,12
5,ABQ,BWI,12
8,ABQ,DAL,12
10,ABQ,DEN,12
12,ABQ,HOU,12
14,ABQ,LAS,12
15,ABQ,LAX,12
17,ABQ,LGB,12
18,ABQ,MCI,12


In [146]:
total_routes = HISTORICAL_ACTIVE_MONTHS.shape[0]

eligible_6 = HISTORICAL_ACTIVE_MONTHS[
    HISTORICAL_ACTIVE_MONTHS["HISTORICAL_ACTIVE_MONTHS"] >= 6
].shape[0]

eligible_6_percentage = (eligible_6 / total_routes) * 100

print("Total routes:", total_routes)
print("Routes with 6+ prior active months:", eligible_6)
print("Percentage:", eligible_6_percentage)

Total routes: 3065
Routes with 6+ prior active months: 1817
Percentage: 59.28221859706362


In [147]:
total_routes_2026_01 = HISTORICAL_ACTIVE_MONTHS_2026_01.shape[0]

eligible_6_2026_01 = HISTORICAL_ACTIVE_MONTHS_2026_01[
    HISTORICAL_ACTIVE_MONTHS_2026_01["HISTORICAL_ACTIVE_MONTHS_2026_01"] >= 6
].shape[0]

eligible_6_2026_01_percentage = (
    eligible_6_2026_01 / total_routes_2026_01
) * 100

print("Total routes:", total_routes_2026_01)
print("Routes with 6+ prior active months:", eligible_6_2026_01)
print("Percentage:", eligible_6_2026_01_percentage)

Total routes: 3942
Routes with 6+ prior active months: 2083
Percentage: 52.84119736174531


In [148]:
recent_routes_2026_01 = wn_route_months[
    (wn_route_months["YEAR MONTH"] >= pd.Period("2025-10")) &
    (wn_route_months["YEAR MONTH"] < pd.Period("2026-01"))
][["ORIGIN", "DEST"]].drop_duplicates()

In [149]:
eligible_6_routes_2026_01 = HISTORICAL_ACTIVE_MONTHS_2026_01[
    HISTORICAL_ACTIVE_MONTHS_2026_01["HISTORICAL_ACTIVE_MONTHS_2026_01"] >= 6
]

eligible_6_recent_2026_01 = eligible_6_routes_2026_01.merge(
    recent_routes_2026_01,
    on=["ORIGIN", "DEST"],
    how="inner"
)

print("6+ historical active months:", len(eligible_6_routes_2026_01))
print("Also active in previous 3 months:", len(eligible_6_recent_2026_01))
print(
    "Percentage:",
    len(eligible_6_recent_2026_01) / len(eligible_6_routes_2026_01) * 100
)

6+ historical active months: 2083
Also active in previous 3 months: 1851
Percentage: 88.86221795487278


In [150]:
eligible_12_routes_2026_01 = HISTORICAL_ACTIVE_MONTHS_2026_01[
    HISTORICAL_ACTIVE_MONTHS_2026_01["HISTORICAL_ACTIVE_MONTHS_2026_01"] >= 12
]

eligible_12_recent_2026_01 = eligible_12_routes_2026_01.merge(
    recent_routes_2026_01,
    on=["ORIGIN", "DEST"],
    how="inner"
)

print("12+ historical active months:", len(eligible_12_routes_2026_01))
print("Also active in previous 3 months:", len(eligible_12_recent_2026_01))
print(
    "Percentage:",
    len(eligible_12_recent_2026_01) / len(eligible_12_routes_2026_01) * 100
)

12+ historical active months: 1768
Also active in previous 3 months: 1697
Percentage: 95.98416289592761


In [151]:
eligible_6_recent_percentage_all = (
    len(eligible_6_recent_2026_01) / total_routes_2026_01
) * 100

eligible_12_recent_percentage_all = (
    len(eligible_12_recent_2026_01) / total_routes_2026_01
) * 100

print("6+ history + recent activity:", eligible_6_recent_percentage_all)
print("12+ history + recent activity:", eligible_12_recent_percentage_all)

6+ history + recent activity: 46.9558599695586
12+ history + recent activity: 43.049213597158804


In [152]:
recent_routes_1m_2026_01 = wn_route_months[
    (wn_route_months["YEAR MONTH"] >= pd.Period("2025-12")) &
    (wn_route_months["YEAR MONTH"] < pd.Period("2026-01"))
][["ORIGIN", "DEST"]].drop_duplicates()

eligible_6_recent_1m_2026_01 = eligible_6_routes_2026_01.merge(
    recent_routes_1m_2026_01,
    on=["ORIGIN", "DEST"],
    how="inner"
)

print("6+ historical active months:", len(eligible_6_routes_2026_01))
print("Also active in previous 1 month:", len(eligible_6_recent_1m_2026_01))
print(
    "Percentage:",
    len(eligible_6_recent_1m_2026_01) / len(eligible_6_routes_2026_01) * 100
)

6+ historical active months: 2083
Also active in previous 1 month: 1755
Percentage: 84.2534805568891


In [153]:
recent_routes_6m_2026_01 = wn_route_months[
    (wn_route_months["YEAR MONTH"] >= pd.Period("2025-07")) &
    (wn_route_months["YEAR MONTH"] < pd.Period("2026-01"))
][["ORIGIN", "DEST"]].drop_duplicates()

eligible_6_recent_6m_2026_01 = eligible_6_routes_2026_01.merge(
    recent_routes_6m_2026_01,
    on=["ORIGIN", "DEST"],
    how="inner"
)

print("6+ historical active months:", len(eligible_6_routes_2026_01))
print("Also active in previous 6 months:", len(eligible_6_recent_6m_2026_01))
print(
    "Percentage:",
    len(eligible_6_recent_6m_2026_01) / len(eligible_6_routes_2026_01) * 100
)

6+ historical active months: 2083
Also active in previous 6 months: 1974
Percentage: 94.76716274603938


In [154]:
eligible_6_routes_2025_01 = HISTORICAL_ACTIVE_MONTHS[
    HISTORICAL_ACTIVE_MONTHS["HISTORICAL_ACTIVE_MONTHS"] >= 6
]

recent_routes_3m_2025_01 = wn_route_months[
    (wn_route_months["YEAR MONTH"] >= pd.Period("2024-10")) &
    (wn_route_months["YEAR MONTH"] < pd.Period("2025-01"))
][["ORIGIN", "DEST"]].drop_duplicates()

eligible_6_recent_3m_2025_01 = eligible_6_routes_2025_01.merge(
    recent_routes_3m_2025_01,
    on=["ORIGIN", "DEST"],
    how="inner"
)

print("6+ historical active months:", len(eligible_6_routes_2025_01))
print(
    "Also active in previous 3 months:",
    len(eligible_6_recent_3m_2025_01)
)
print(
    "Percentage:",
    len(eligible_6_recent_3m_2025_01)
    / len(eligible_6_routes_2025_01)
    * 100
)

6+ historical active months: 1817
Also active in previous 3 months: 1739
Percentage: 95.7072096862961


In [155]:
recent_3m_count = len(eligible_6_recent_2026_01)
recent_6m_count = len(eligible_6_recent_6m_2026_01)

recent_3m_percentage = (
    recent_3m_count / total_routes_2026_01
) * 100

recent_6m_percentage = (
    recent_6m_count / total_routes_2026_01
) * 100

print("Total observed routes:", total_routes_2026_01)

print(
    "6+ history + active in previous 3 months:",
    recent_3m_count,
    f"({recent_3m_percentage:.2f}%)"
)

print(
    "6+ history + active in previous 6 months:",
    recent_6m_count,
    f"({recent_6m_percentage:.2f}%)"
)

print(
    "Additional routes kept by 6-month window:",
    recent_6m_count - recent_3m_count
)

print(
    "Percentage-point difference:",
    recent_6m_percentage - recent_3m_percentage
)

Total observed routes: 3942
6+ history + active in previous 3 months: 1851 (46.96%)
6+ history + active in previous 6 months: 1974 (50.08%)
Additional routes kept by 6-month window: 123
Percentage-point difference: 3.1202435312024335


In [156]:
recent_routes_6m_2025_01 = wn_route_months[
    (wn_route_months["YEAR MONTH"] >= pd.Period("2024-07")) &
    (wn_route_months["YEAR MONTH"] < pd.Period("2025-01"))
][["ORIGIN", "DEST"]].drop_duplicates()

eligible_6_recent_6m_2025_01 = eligible_6_routes_2025_01.merge(
    recent_routes_6m_2025_01,
    on=["ORIGIN", "DEST"],
    how="inner"
)

recent_3m_count_2025_01 = len(eligible_6_recent_3m_2025_01)
recent_6m_count_2025_01 = len(eligible_6_recent_6m_2025_01)

recent_3m_percentage_2025_01 = (
    recent_3m_count_2025_01 / HISTORICAL_ACTIVE_MONTHS.shape[0]
) * 100

recent_6m_percentage_2025_01 = (
    recent_6m_count_2025_01 / HISTORICAL_ACTIVE_MONTHS.shape[0]
) * 100

print("Total observed routes:", HISTORICAL_ACTIVE_MONTHS.shape[0])

print(
    "6+ history + active in previous 3 months:",
    recent_3m_count_2025_01,
    f"({recent_3m_percentage_2025_01:.2f}%)"
)

print(
    "6+ history + active in previous 6 months:",
    recent_6m_count_2025_01,
    f"({recent_6m_percentage_2025_01:.2f}%)"
)

print(
    "Additional routes kept by 6-month window:",
    recent_6m_count_2025_01 - recent_3m_count_2025_01
)

print(
    "Percentage-point difference:",
    recent_6m_percentage_2025_01 - recent_3m_percentage_2025_01
)

Total observed routes: 3065
6+ history + active in previous 3 months: 1739 (56.74%)
6+ history + active in previous 6 months: 1800 (58.73%)
Additional routes kept by 6-month window: 61
Percentage-point difference: 1.9902120717781386


In [157]:
print("Percentage of 12+ historical active months + active in previous 3 months:", len(eligible_12_recent_2026_01) / total_routes_2026_01 * 100)
print("Percentage of 6+ historical active months + active in previous 3 months:", len(eligible_6_recent_2026_01) / total_routes_2026_01 * 100)


Percentage of 12+ historical active months + active in previous 3 months: 43.049213597158804
Percentage of 6+ historical active months + active in previous 3 months: 46.9558599695586


In [158]:
eligible_12_routes_2025_01 = HISTORICAL_ACTIVE_MONTHS[
    HISTORICAL_ACTIVE_MONTHS["HISTORICAL_ACTIVE_MONTHS"] >= 12
]

recent_routes_3m_2025_01 = wn_route_months[
    (wn_route_months["YEAR MONTH"] >= pd.Period("2024-10")) &
    (wn_route_months["YEAR MONTH"] < pd.Period("2025-01"))
][["ORIGIN", "DEST"]].drop_duplicates()

eligible_12_recent_3m_2025_01 = eligible_12_routes_2025_01.merge(
    recent_routes_3m_2025_01,
    on=["ORIGIN", "DEST"],
    how="inner"
)

print("6+ history + 3-month recency:", len(eligible_6_recent_3m_2025_01))
print("12+ history + 3-month recency:", len(eligible_12_recent_3m_2025_01))
print("difference:", len(eligible_6_recent_3m_2025_01) - len(eligible_12_recent_3m_2025_01))

6+ history + 3-month recency: 1739
12+ history + 3-month recency: 1433
difference: 306


### Leakage-Safe Route Eligibility Rule

The exploratory route-recurrence analysis showed that using a fixed full-period threshold such as `ACTIVE_MONTHS >= 24` would introduce look-ahead leakage during historical backtesting because it uses information from future months.

To avoid this, route eligibility must be determined separately at each forecast origin using only information available before the forecast month.

Two historical-activity thresholds were investigated:

- at least 6 prior active months
- at least 12 prior active months

A recent-activity condition was also investigated to avoid keeping routes that had sufficient historical observations but were no longer recently active.

For a January 2025 forecast origin:

- 6+ historical active months + activity in the previous 3 months:
  - 1,739 eligible routes
  - 56.74% of 3,065 observed routes
- 12+ historical active months + activity in the previous 3 months:
  - 1,433 eligible routes
  - 46.75% of 3,065 observed routes

For a January 2026 forecast origin:

- 6+ historical active months + activity in the previous 3 months:
  - 1,851 eligible routes
  - 46.96% of 3,942 observed routes
- 12+ historical active months + activity in the previous 3 months:
  - 1,697 eligible routes
  - 43.05% of 3,942 observed routes

Recent-activity windows of 1, 3, and 6 months were also compared. A 1-month window was considered too restrictive because a route could remain relevant even if it did not operate in the immediately preceding month. A 6-month window retained only a modest number of additional routes while allowing considerably older activity to qualify. The 3-month window provided a reasonable balance between route retention and evidence of current relevance.

The selected leakage-safe route eligibility rule is therefore:

> At forecast month `t`, a WN directional route is eligible only if:
>
> 1. it has been active in at least 6 historical months before `t`, and
> 2. it has been active at least once during the 3 calendar months immediately preceding `t`.

This rule is leakage-safe because both conditions use only information available before the forecast origin.

The 6-month historical threshold was preferred over 12 months because the 12-month requirement was substantially more restrictive, especially at earlier forecast origins. At the January 2025 forecast origin, requiring 12 active months effectively required the route to have been active during all 12 available historical months, which imposed an unnecessarily strict continuity condition.

This eligibility rule is intended only to determine which routes have sufficient and sufficiently recent history for later modeling. It does not yet define the final route-month modeling dataset, lag features, target construction, or model inputs.

In [159]:
def get_eligible_routes(wn_route_months, forecast_origin):
    HISTORICAL_ACTIVE_MONTHS = wn_route_months[
        wn_route_months["YEAR MONTH"] < forecast_origin
    ].groupby(["ORIGIN", "DEST"])["YEAR MONTH"].agg('count').reset_index(name="HISTORICAL_ACTIVE_MONTHS")
    
    eligible_routes = HISTORICAL_ACTIVE_MONTHS[
        HISTORICAL_ACTIVE_MONTHS["HISTORICAL_ACTIVE_MONTHS"] >= 6
    ]
    
    recent_routes = wn_route_months[
        (wn_route_months["YEAR MONTH"] >= forecast_origin - 3) &
        (wn_route_months["YEAR MONTH"] < forecast_origin)
    ][["ORIGIN", "DEST"]].drop_duplicates()
    
    eligible_recent_routes = eligible_routes.merge(
        recent_routes,
        on=["ORIGIN", "DEST"],
        how="inner"
    )
    
    return eligible_recent_routes

In [160]:
eligible_2025_01 = get_eligible_routes(
    wn_route_months,
    pd.Period("2025-01")
)

eligible_2026_01 = get_eligible_routes(
    wn_route_months,
    pd.Period("2026-01")
)

print("2025-01:", len(eligible_2025_01))
print("2026-01:", len(eligible_2026_01))

2025-01: 1739
2026-01: 1851


### Route Eligibility Implementation Validation

The selected leakage-safe route eligibility rule was implemented as a reusable function.

For a forecast month `t`, a WN directional route is eligible if:

1. it has at least 6 historical active months before `t`, and
2. it was active at least once during the 3 calendar months immediately preceding `t`.

The function was validated against previously calculated exploratory results:

- Forecast origin `2025-01`: 1,739 eligible routes
- Forecast origin `2026-01`: 1,851 eligible routes

These results matched the independently calculated eligibility counts, confirming that the reusable implementation correctly applies the selected rule using only information available before each forecast origin.

The function defines route eligibility only. It does not construct the final modeling dataset, create lag features, or define the prediction target.

## Phase 4.2 Conclusion

Phase 4.2 completed the Pandas/NumPy processing and carrier investigation required before construction of the modeling dataset.

### Carrier Selection

Southwest Airlines (`WN`) was selected as the project carrier based on its combination of:

- large route-network breadth,
- high passenger volume,
- substantial long-running route coverage, and
- strong historical continuity.

Across the January 2024 through May 2026 historical period, WN operated 4,217 observed directional routes and carried approximately 414.12 million passengers in the active scheduled-passenger data.

The exploratory recurrence analysis identified 1,515 WN routes active in at least 24 of the 29 observed months. Of these, 1,309 were active during all 29 months, and approximately 91.29% had continuous histories without internal gaps.

The `ACTIVE_MONTHS >= 24` definition was used only for exploratory stability analysis and will not be used as a modeling eligibility rule because it relies on information from the full historical period and would introduce look-ahead leakage during historical backtesting.

### Leakage-Safe Route Eligibility

A leakage-safe eligibility rule was developed using only information available before each forecast origin.

For forecast month `t`, a WN directional route is eligible if:

1. it has been active in at least 6 historical months before `t`, and
2. it was active at least once during the 3 calendar months immediately preceding `t`.

The 6-month historical requirement was preferred over a 12-month requirement because the 12-month threshold was unnecessarily restrictive, particularly at earlier forecast origins.

A 3-month recent-activity window was selected instead of a 1-month or 6-month window because it provided a reasonable balance between avoiding stale routes and retaining currently relevant routes.

The rule was implemented as a reusable function and independently validated:

- Forecast origin `2025-01`: 1,739 eligible routes
- Forecast origin `2026-01`: 1,851 eligible routes

The validation results matched the earlier exploratory calculations.

### Phase Boundary

Phase 4.2 defines the selected carrier and the leakage-safe route-eligibility logic.

It does not yet:

- construct the final route-month modeling dataset,
- define the prediction target,
- create lag or rolling features,
- perform feature engineering, or
- train forecasting models.

Those steps belong to subsequent phases.